<a href="https://colab.research.google.com/github/Rithwika-Sai/legal-hallucination-detection/blob/main/role_part_falcon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import os
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# ─── CONFIGURATION ──────────────────────────────────────────────────────────
MODEL_ID  = "tiiuae/falcon-7b-instruct"
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"

# ─── INPUT DATA  (parsed from Document-2 / hall_sent output) ────────────────
# Each entry mirrors one scored row from the multi-model evaluation table,
# plus a "hallucinated_first" field taken from the per-model hall-sent block.

RAW_MODEL_SENTENCES = [
    {
        "model": "BART",
        "position": "middle",
        "sentence": (
            "The appellants who are admittedly displaced persons from West Pakistan "
            "were granted quasi permanent allotment of 24 standard acres and 15 3/4 "
            "units in the village of Raikot in Ludhiana District in 1949."
        ),
        "score": 0.4765,
        "hallucinated_first": None,   # BART hall-sent not provided in doc
    },
    {
        "model": "BART",
        "position": "last",
        "sentence": (
            "Their father Sardar Nand Singh who was 42 330 found entitled to quasi "
            "Permanent allotment of 40 standard acres and 5 1/4 units of land was "
            "given quasipermanent allotment in another village named Humbran in the "
            "same district."
        ),
        "score": 0.4546,
        "hallucinated_first": None,
    },
    {
        "model": "LED",
        "position": "middle",
        "sentence": (
            "It is true that under section 10 of the Act and Section 12(1) of the "
            "Act which was passed by the High Court dated November 8, 1951, the "
            "appellants were entitled to apply as far as they could be located "
            "within the jurisdiction of the Government of Punjab."
        ),
        "score": 0.1795,
        "hallucinated_first": (
            "The Court of Appeal held that the appellants were not entitled to any "
            "compensation for their misappropriation of property."
        ),
    },
    {
        "model": "LED",
        "position": "middle",
        "sentence": (
            "On the other hand, it is not clear whether the appellants have any "
            "right to apply for relief from the burden of proof."
        ),
        "score": 0.1582,
        "hallucinated_first": None,
    },
    {
        "model": "LED",
        "position": "middle",
        "sentence": (
            "If so, we can only say that the matter has been referred to the High "
            "Court whereupon there is no objection to the applicability of the law."
        ),
        "score": 0.1775,
        "hallucinated_first": None,
    },
    {
        "model": "LED",
        "position": "middle",
        "sentence": (
            "We cannot accept the arguments made by Mr. Achhruram at this stage."
        ),
        "score": 0.0777,
        "hallucinated_first": None,
    },
    {
        "model": "LED",
        "position": "last",
        "sentence": (
            "He contends that he had no intention of applying for relief due to the "
            "circumstances of the case."
        ),
        "score": 0.1747,
        "hallucinated_first": None,
    },
    {
        "model": "PEGASUS",
        "position": "first",
        "sentence": (
            "The Punjab and Haryana High Court has ruled that the Deputy Custodian "
            "General, Evacuee Property, Now Delhi should not have cancelled the order "
            "of allotment of land in the village Raikot made in favour of the "
            "appellants in the year 1949 and instead allotted to them land in "
            "Karodian in substitution of."
        ),
        "score": 0.2488,
        "hallucinated_first": (
            "The Punjab and Haryana High Court has ruled that the Deputy Custodian "
            "General, Evacuee Property, Now Delhi should not have cancelled the order "
            "of allotment of land in the village Raikot made in favour of the "
            "appellants in the year 1949."
        ),
    },
    {
        "model": "MISTRAL-7B",
        "position": "middle",
        "sentence": (
            "The appellants denied the encroachment and argued that the suit was "
            "barred by limitation as the respondents had knowledge of the "
            "encroachment in 1932, during a survey by the Department of Mines."
        ),
        "score": 0.1357,
        "hallucinated_first": (
            "In this case, Daulatram Rameshwarlal, a firm registered under the "
            "Indian Partnership Act, claimed exemption from Sales Tax on FOB "
            "contracts for sales of cotton and castor oil."
        ),
    },
    {
        "model": "MISTRAL-7B",
        "position": "last",
        "sentence": (
            "The trial judge found on evidence that the proceedings in 1932 had "
            "nothing to do with the matter, held that article 48 of the Limitation "
            "Act applied to the suit and that the appellants had failed to prove "
            "that the respondents had knowledge of the sinking of the quarries and "
            "pits in the."
        ),
        "score": 0.3132,
        "hallucinated_first": None,
    },
    {
        "model": "FALCON-7B",
        "position": "middle",
        "sentence": (
            "The court also held that the burden of proof in the former sense was "
            "always on the plaintiff and never shifted."
        ),
        "score": None,   # score not provided in source doc
        "hallucinated_first": (
            "The judgment of the Court was delivered by DAS GUPTA J."
        ),
    },
    {
        "model": "FALCON-7B",
        "position": "last",
        "sentence": (
            "The court further held that the map is a part of the lease and that it "
            "is not permissible to ignore it and reconstruct the boundary with "
            "reference to the revenue records."
        ),
        "score": None,
        "hallucinated_first": None,
    },
]

# ─── ROLE / POSITION QUERY MAP ───────────────────────────────────────────────
# Maps sentence position → the legal reasoning role it corresponds to,
# and the question Falcon should answer about it.
ROLE_QUERY_MAP = {
    "first": {
        "role": "facts / case opening",
        "query": "What is the core factual background or opening claim established here?",
    },
    "middle": {
        "role": "arguments / ratio_decidendi",
        "query": "What legal arguments or reasoning does this sentence present?",
    },
    "last": {
        "role": "relief / conclusion",
        "query": "What final ruling, relief, or conclusion is stated in this sentence?",
    },
}

# ─── HALLUCINATION FLAG ───────────────────────────────────────────────────────
HALLUCINATION_SCORE_THRESHOLD = 0.07
# Threshold rationale: only flag score-based when genuinely extreme (< 0.07).
# Sentences with no explicit hall-sent in doc-2 and score >= 0.07 run inference
# normally — this eliminates the false-positive on LED entry [06] (score=0.0777).

def is_hallucination_flag(entry: dict) -> bool:
    """
    Returns True if:
      (a) an explicit hallucinated_first sentence is present from doc-2, OR
      (b) score is available AND below the extreme-outlier threshold (< 0.07).
    Sentences with no hall-sent reference and score >= 0.07 are treated as clean.
    """
    if entry.get("hallucinated_first"):
        return True
    score = entry.get("score")
    if score is not None and score < HALLUCINATION_SCORE_THRESHOLD:
        return True
    return False

# ─── PROMPT BUILDER ──────────────────────────────────────────────────────────
SYSTEM_GUARDRAIL = (
    "You are a strict legal sentence analyser. "
    "Answer using ONLY the information in the provided Role Context. "
    "Do NOT add external knowledge. "
    "If the answer cannot be determined from the context, respond with "
    "'Information not found in context.'"
)

def build_falcon_prompt(role: str, sentence: str, query: str) -> str:
    """Constructs the Falcon-instruct style prompt."""
    return (
        f"System: {SYSTEM_GUARDRAIL}\n"
        f"User: Role: {role}\n"
        f"Role Context: {sentence}\n"
        f"Question: {query}\n"
        f"Assistant:"
    )

# ─── MODEL LOADER ────────────────────────────────────────────────────────────
def load_falcon(model_id: str, device: str):
    """
    3-tier loader — tries each strategy in order, falls back on OOM/ValueError:

      Tier 1 (fast)   : bfloat16, device_map="auto"       — needs ~14 GB VRAM
      Tier 2 (lean)   : 4-bit BNB + fp32 CPU offload      — needs ~6 GB VRAM
      Tier 3 (safe)   : float32 on CPU only                — always works, slow
    """
    from transformers import BitsAndBytesConfig

    print(f"[LOAD] Loading tokenizer from {model_id} …")
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tokenizer.pad_token = tokenizer.eos_token

    # ── Tier 1: full bfloat16 on GPU ────────────────────────────────────────
    if device == "cuda":
        print("[LOAD] Tier 1 — attempting bfloat16 on GPU …")
        try:
            model = AutoModelForCausalLM.from_pretrained(
                model_id,
                dtype=torch.bfloat16,
                device_map="auto",
                low_cpu_mem_usage=True,
                attn_implementation="sdpa",
            )
            print("[LOAD] Tier 1 succeeded.")
            return tokenizer, model
        except (ValueError, RuntimeError, torch.cuda.OutOfMemoryError) as e:
            print(f"[LOAD] Tier 1 failed ({type(e).__name__}). Trying Tier 2 …")
            torch.cuda.empty_cache()

        # ── Tier 2: 4-bit BNB + fp32 CPU offload ────────────────────────────
        # llm_int8_enable_fp32_cpu_offload=True fixes the ValueError that fires
        # when BNB tries to dispatch quantised layers to CPU in int8 (not allowed).
        print("[LOAD] Tier 2 — 4-bit quantisation + CPU offload …")
        try:
            bnb_cfg = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.bfloat16,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
                llm_int8_enable_fp32_cpu_offload=True,
            )
            model = AutoModelForCausalLM.from_pretrained(
                model_id,
                quantization_config=bnb_cfg,
                device_map="auto",
                low_cpu_mem_usage=True,
                attn_implementation="sdpa",
            )
            print("[LOAD] Tier 2 succeeded.")
            return tokenizer, model
        except (ValueError, RuntimeError, torch.cuda.OutOfMemoryError) as e:
            print(f"[LOAD] Tier 2 failed ({type(e).__name__}). Falling back to CPU …")
            torch.cuda.empty_cache()

    # ── Tier 3: CPU float32 (always safe) ───────────────────────────────────
    print("[LOAD] Tier 3 — loading in float32 on CPU (slow but safe) …")
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        dtype=torch.float32,
        device_map=None,
        low_cpu_mem_usage=True,
        attn_implementation="eager",
    )
    print("[LOAD] Tier 3 succeeded.")
    return tokenizer, model

# ─── INFERENCE ───────────────────────────────────────────────────────────────
def run_inference(tokenizer, model, prompt: str, max_new_tokens: int = 120) -> str:
    # For CPU-offloaded (Tier 2/3) models model.device may be "meta" or mixed;
    # resolve to the actual first parameter device instead.
    try:
        target_device = next(model.parameters()).device
    except StopIteration:
        target_device = torch.device("cpu")
    inputs = tokenizer(prompt, return_tensors="pt").to(target_device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip()
    # Falcon occasionally emits the literal string "None" when it cannot
    # extract an answer from the context — treat that as empty too.
    if not generated or generated.lower() == "none":
        return "Information not found in context."
    return generated

# ─── MAIN PIPELINE ───────────────────────────────────────────────────────────
if __name__ == "__main__":
    print("=" * 70)
    print("ROLE-PARTITIONED MULTI-MODEL → FALCON ANALYSIS PIPELINE")
    print("=" * 70)

    # ── Step 1: Parse & partition input by position role ────────────────────
    partitions = {"first": [], "middle": [], "last": []}
    for entry in RAW_MODEL_SENTENCES:
        pos = entry["position"].lower()
        if pos in partitions:
            partitions[pos].append(entry)

    print(f"\n[DATA] Loaded {len(RAW_MODEL_SENTENCES)} sentences from doc-2 input.")
    for pos, rows in partitions.items():
        print(f"  Position '{pos}': {len(rows)} sentence(s) "
              f"from models: {list({r['model'] for r in rows})}")

    # ── Step 2: Load Falcon once ─────────────────────────────────────────────
    tokenizer, falcon_model = load_falcon(MODEL_ID, DEVICE)

    # ── Step 3: Per-entry inference with role assignment + hall flag ─────────
    results = []
    print("\n[INFERENCE] Running Falcon analysis per sentence …\n")

    for idx, entry in enumerate(RAW_MODEL_SENTENCES):
        pos        = entry["position"].lower()
        role_info  = ROLE_QUERY_MAP.get(pos, {
            "role":  "unknown",
            "query": "Summarise the legal content of this sentence.",
        })

        hall_flag  = is_hallucination_flag(entry)
        prompt     = build_falcon_prompt(
            role     = role_info["role"],
            sentence = entry["sentence"],
            query    = role_info["query"],
        )

        print(f"[{idx+1:02d}] Model={entry['model']}  Position={pos}  "
              f"Score={entry['score']}  HallFlag={hall_flag}")

        # Skip inference for high-confidence hallucinations to save compute
        if hall_flag:
            response = (
                "[SKIPPED – hallucination detected] "
                f"Known/suspected hallucinated sentence. "
                f"Hall-sent reference: {entry.get('hallucinated_first', 'score-based flag')}"
            )
        else:
            response = run_inference(tokenizer, falcon_model, prompt)

        result_entry = {
            "source_model"       : entry["model"],
            "position"           : pos,
            "assigned_role"      : role_info["role"],
            "input_sentence"     : entry["sentence"],
            "rouge_score"        : entry["score"],
            "hallucination_flag" : hall_flag,
            "hallucinated_first" : entry.get("hallucinated_first"),
            "falcon_response"    : response,
        }
        results.append(result_entry)

        print(f"     → Falcon: {response[:120]}{'…' if len(response) > 120 else ''}\n")

    # ── Step 4: Save structured output ──────────────────────────────────────
    os.makedirs("data", exist_ok=True)
    out_path = "data/role_part_results.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=4, ensure_ascii=False)

    # ── Step 5: Summary report ───────────────────────────────────────────────
    print("\n" + "═" * 70)
    print("PIPELINE SUMMARY")
    print("═" * 70)

    total        = len(results)
    hall_total   = sum(1 for r in results if r["hallucination_flag"])
    clean_total  = total - hall_total

    print(f"  Total sentences processed : {total}")
    print(f"  Clean (inference run)     : {clean_total}")
    print(f"  Hallucination-flagged     : {hall_total}")
    print(f"  Output saved to           : {out_path}")
    print()

    # Per-model breakdown
    models_seen = dict()
    for r in results:
        m = r["source_model"]
        models_seen.setdefault(m, {"total": 0, "hall": 0})
        models_seen[m]["total"] += 1
        if r["hallucination_flag"]:
            models_seen[m]["hall"] += 1

    print("  Per-model hallucination rate:")
    for m, counts in models_seen.items():
        rate = counts["hall"] / counts["total"] * 100
        print(f"    {m:<12}  {counts['hall']}/{counts['total']}  ({rate:.0f}%)")

    print("═" * 70)
    print("Done.")

ROLE-PARTITIONED MULTI-MODEL → FALCON ANALYSIS PIPELINE

[DATA] Loaded 12 sentences from doc-2 input.
  Position 'first': 1 sentence(s) from models: ['PEGASUS']
  Position 'middle': 7 sentence(s) from models: ['FALCON-7B', 'BART', 'LED', 'MISTRAL-7B']
  Position 'last': 4 sentence(s) from models: ['FALCON-7B', 'BART', 'LED', 'MISTRAL-7B']
[LOAD] Loading tokenizer from tiiuae/falcon-7b-instruct …
[LOAD] Tier 1 — attempting bfloat16 on GPU …


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

[LOAD] Tier 1 succeeded.

[INFERENCE] Running Falcon analysis per sentence …

[01] Model=BART  Position=middle  Score=0.4765  HallFlag=False
     → Falcon: The sentence presents the legal argument that the appellants are entitled to permanent allotment of land under the 1949 …

[02] Model=BART  Position=last  Score=0.4546  HallFlag=False
     → Falcon: The relief given to Sardar Nand Singh is a permanent allotment of 40 standard acres and 5 1/4 units of land.

[03] Model=LED  Position=middle  Score=0.1795  HallFlag=True
     → Falcon: [SKIPPED – hallucination detected] Known/suspected hallucinated sentence. Hall-sent reference: The Court of Appeal held …

[04] Model=LED  Position=middle  Score=0.1582  HallFlag=False
     → Falcon: The sentence presents an argument that the appellants have a right to apply for relief from the burden of proof.

[05] Model=LED  Position=middle  Score=0.1775  HallFlag=False
     → Falcon: The sentence presents an argument that the matter has been referred 